# Image deconvolution of the 26Al (1809 keV) and 511 keV lines with the DC4 three-month data

This tutorial makes all-sky images of the two brightest gamma-ray lines in the COSI Data Challenge 4 (DC4) mock data set: the 26Al line at 1809 keV and the positron-annihilation line at 511 keV. It uses two algorithms of `cosipy.image_deconvolution`:

* **Richardson-Lucy** (`RLadvanced`: accelerated ML-EM with response weighting and smoothing of the update). It is regularised by the number of iterations.
* **MREM** (multi-resolution regularised EM, Knoedlseder+99). Only the part of each correction that is significant at some angular scale is applied, so the iteration converges instead of fitting noise.

The setup is the same for both lines:

* **Data space:** binned Compton data space (CDS) in Galactic coordinates: 60 Compton-scattering-angle bins of 3 deg (Phi) x HEALPix nside 16 scattered directions (PsiChi), the whole three months in one time bin. This is the binning of the DC4 line responses.
* **Image response:** computed with cosipy from the DC4 detector response and the DC4 orientation file. For every image pixel it is the point-source response of the pixel centre, *with Earth occultation*, merged into one Galactic response (`FullDetectorResponse.get_point_source_response_per_image_pixel` and `merge_psr_to_extended_source_response`).
* **Background:**
  * *26Al:* data-driven. Three sideband windows of the mock data next to the line (continuum below and above it, and the window with the instrumental lines at 1764 and 1779 keV) are smoothed over the scattered direction, and each gets its own normalisation, fitted together with the image.
  * *511 keV:* the background model released with DC4 for the 511 keV line window, used at its own normalisation.

Steps:
1. Read the mock data once, select the two energy ranges and bin them in the Galactic CDS (`BinnedData`).
2. Compute the image responses of both lines.
3. Build the background models.
4. Run RL and MREM for each line (`ImageDeconvolution`).
5. Compare the images, their longitude and latitude profiles and the convergence.

**Runtime and resources.** The heavy steps cache their results in `data_dir`, so a second run only repeats the deconvolutions. Measured on a 32-core Linux machine with 8 worker processes:

| step | first run | cached |
|---|---|---|
| download (11.4 GB, depends on the connection) | - | skipped |
| read the mock data, select both energy ranges and bin them (needs about 37 GB of memory) | 2 min | < 1 s |
| image response 26Al (3072 point-source responses) | 10 min | 10 s |
| image response 511 keV (3072 point-source responses) | 10 min | 10 s |
| the four deconvolutions | listed by the last cell | same |

The image responses are computed for `NSIDE_IMAGE = 16` (3.7 deg pixels, 3072 sky positions per line), the resolution of the DC4 responses. `NSIDE_IMAGE = 8` is about four times faster, but too coarse for the narrow Galactic plane.

In [ ]:
import logging
import sys
logger = logging.getLogger('cosipy')
logger.setLevel(logging.INFO)
logger.addHandler(logging.StreamHandler(sys.stdout))

In [ ]:
import time
import multiprocessing
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

import numpy as np
import healpy as hp
import matplotlib.pyplot as plt
from matplotlib.colors import PowerNorm
import astropy.units as u
from histpy import Histogram

import cosipy
from cosipy.util import fetch_wasabi_file
from cosipy.data_io import BinnedData
from cosipy.spacecraftfile import SpacecraftHistory
from cosipy.response import FullDetectorResponse, ExtendedSourceResponse
from cosipy.image_deconvolution import DataIF_COSI_DC2, ImageDeconvolution

%matplotlib inline
plt.rcParams['figure.dpi'] = 72
logger.info(f'cosipy {cosipy.__version__}')

### Settings

<span style="color:red">Set `data_dir` to the directory where the input files are downloaded and the intermediate results are cached. It needs about 11 GB for the inputs, 0.6 GB for the cached event selections and 2.3 GB per line for the image responses.</span>

In [ ]:
data_dir = Path('./')

NSIDE_IMAGE = 16       # image pixels = sky positions of the point-source responses (must match the parameter files)
N_WORKERS = 8          # processes for the point-source responses (1: serial)
NSIDE_SCATT = 32       # resolution of the spacecraft-attitude map used for the point-source responses

data_dir.mkdir(parents=True, exist_ok=True)
runtime = {}   # measured wall-clock times [s]

## 0. Input files

All inputs are public DC4 files on wasabi:
- `COSI-SMEX/DC4/Data/Mock_Dataset/dc4_mock_dataset_3months_unbinned_data_filtered_with_SAAcut_time_ordered.fits.gz` (10.8 GB): the DC4 mock data, three months, unbinned (all sources plus background). Only needed once: the two energy selections are cached in `data_dir` (section 1), and the download is skipped once the caches exist.
- `COSI-SMEX/DC4/Data/Orientation/DC4_final_530km_3_month_with_slew_15sbins_GalacticEarth_SAA.fits` (38 MB): the spacecraft orientation and location (for the point-source responses and the Earth occultation).
- `COSI-SMEX/DC4/Data/Responses/Response26Al.o4.e1805_1812.s10036231691364.m1045.filtered.nonsparse.binnedimaging.imagingresponse.h5` (287 MB): the detector response of the 26Al line window, 1805-1812 keV.
- `COSI-SMEX/DC4/Data/Responses/Response511.o4.e509_513.s20881894470591.m2555.filtered.nonsparse.binnedimaging.imagingresponse.h5` (287 MB): the detector response of the 511 keV line window, 509-513 keV.
- `COSI-SMEX/DC4/Data/Backgrounds/gal_DC4_bkg_511_binned_smoothed_fwhm_10.0deg.hdf5` (1.6 MB): the released background model of the 511 keV line window in the Galactic CDS.

`fetch_wasabi_file` skips a file that already exists with the given checksum.

In [ ]:
mock_key = 'COSI-SMEX/DC4/Data/Mock_Dataset/dc4_mock_dataset_3months_unbinned_data_filtered_with_SAAcut_time_ordered.fits.gz'
orientation_key = 'COSI-SMEX/DC4/Data/Orientation/DC4_final_530km_3_month_with_slew_15sbins_GalacticEarth_SAA.fits'
response_keys = {'26Al': 'COSI-SMEX/DC4/Data/Responses/Response26Al.o4.e1805_1812.s10036231691364.m1045.filtered.nonsparse.binnedimaging.imagingresponse.h5',
                 '511keV': 'COSI-SMEX/DC4/Data/Responses/Response511.o4.e509_513.s20881894470591.m2555.filtered.nonsparse.binnedimaging.imagingresponse.h5'}
bkg511_key = 'COSI-SMEX/DC4/Data/Backgrounds/gal_DC4_bkg_511_binned_smoothed_fwhm_10.0deg.hdf5'

mock_file = data_dir / Path(mock_key).name
orientation_file = data_dir / Path(orientation_key).name
response_files = {line: data_dir / Path(key).name for line, key in response_keys.items()}
bkg511_file = data_dir / Path(bkg511_key).name

# Orientation file (38 MB):
fetch_wasabi_file(orientation_key, output=orientation_file, checksum='ca94ff1d7a73c1f41479aaf598807673')
# Response files (287 MB each):
fetch_wasabi_file(response_keys['26Al'], output=response_files['26Al'], checksum='4de7c15f95c62698ff139585f5f5cdd1')
fetch_wasabi_file(response_keys['511keV'], output=response_files['511keV'], checksum='d15cbd03b2cd3d50f3411cd6b8ee362e')
# Released 511 keV background model (1.6 MB):
fetch_wasabi_file(bkg511_key, output=bkg511_file, checksum='1b437be9b4d9e85a230dfe4f4d2ca67e')

## 1. Select the two energy ranges and bin them in the Galactic Compton data space

The mock data contain about 170 million events (19 GB uncompressed). `BinnedData.get_dict_from_fits` reads the compressed file into memory (about 37 GB peak). Then two energy ranges are selected with `select_data_energy` and cached as hdf5:
* 1700-1885 keV for 26Al (the line window and its sidebands);
* 509-513 keV for the 511 keV line window.

`select_data_energy` replaces the arrays of the dictionary it is given. Each selection therefore gets its own shallow copy of the full dictionary, which the selection leaves intact.

The binning is defined in `inputs_DC4_26Al.yaml` and `inputs_DC4_511keV.yaml`: one time bin over the three months, Phi bins of 3 deg and PsiChi HEALPix nside 16 (ring). The energy bins are:
* 26Al: 1700, 1755, 1760, 1784, 1805, 1812, 1822, 1885 keV. These are the low continuum, a gap, the instrumental lines, a gap, the line window, a gap and the high continuum. The gaps keep the line wings and the instrumental-line wings out of the background windows.
* 511 keV: 509-513 keV.

The line windows and the Phi and PsiChi axes are the axes of the responses.

In [ ]:
SELECTIONS = {'26Al': (1700.0, 1885.0), '511keV': (509.0, 513.0)}   # keV, half-open [emin, emax)
BINNING_YAML = {'26Al': 'inputs_DC4_26Al.yaml', '511keV': 'inputs_DC4_511keV.yaml'}

selection_prefix = {line: data_dir / f'DC4_mock_{lo:.0f}_{hi:.0f}keV' for line, (lo, hi) in SELECTIONS.items()}
selection_file = {line: p.with_name(p.name + '.hdf5') for line, p in selection_prefix.items()}
binned_prefix = {line: data_dir / f'DC4_mock_{lo:.0f}_{hi:.0f}keV_galactic_binned' for line, (lo, hi) in SELECTIONS.items()}
binned_file = {line: p.with_name(p.name + '.hdf5') for line, p in binned_prefix.items()}

if not all(f.exists() for f in selection_file.values()):
    # Mock data (10.8 GB):
    fetch_wasabi_file(mock_key, output=mock_file, checksum='762b7ad849b501ddaa31799ec19d14b9')
    t0 = time.perf_counter()
    reader = BinnedData(BINNING_YAML['26Al'])
    full = reader.get_dict_from_fits(mock_file)
    for line, (lo, hi) in SELECTIONS.items():
        reader.cosi_dataset = dict(full)          # shallow copy: the selection replaces the arrays of this copy only
        reader.select_data_energy(lo, hi, output_name=str(selection_prefix[line]))
    del full, reader
    runtime['read and select'] = time.perf_counter() - t0

counts = {}
for line in SELECTIONS:
    if not binned_file[line].exists():
        t0 = time.perf_counter()
        binned = BinnedData(BINNING_YAML[line])
        binned.get_binned_data(unbinned_data=selection_file[line], psichi_binning='galactic', sparse=False,
                               output_name=str(binned_prefix[line]))
        runtime[f'binning {line}'] = time.perf_counter() - t0
    counts[line] = Histogram.open(binned_file[line]).project('Em', 'Phi', 'PsiChi')      # drop the single time bin
    print(line, counts[line].axes['Em'].edges, f'{counts[line].contents.sum():.0f} events')

## 2. Image responses (Galactic, with Earth occultation)

For every image pixel (HEALPix `NSIDE_IMAGE`, Galactic, ring scheme), `FullDetectorResponse.get_point_source_response_per_image_pixel(..., earth_occ=True)` computes the point-source response of the pixel centre from the detector response and the orientation file. The source path through the instrument frame is built from a spacecraft-attitude map (`nside_scatt_map = 32`), using only the times when the position is not behind the Earth. The responses are computed in parallel, and each is written to its own file, so an interrupted run resumes. They are then merged into one Galactic extended-source response with `FullDetectorResponse.merge_psr_to_extended_source_response`.

`DataIF_COSI_DC2` accepts this merged response directly as a precomputed Galactic image response. Its unit is cm2 s, i.e. counts per CDS bin per unit flux in the line window, accumulated over three months.

In [ ]:
def _init_worker(response_file):
    global _drm, _ori
    _drm = FullDetectorResponse.open(response_file, dtype=np.float32)
    _ori = SpacecraftHistory.open(orientation_file)

def _psr_to_file(args):
    """Point-source response of the centre of image pixel ipix, Earth occultation included."""
    ipix, psr_dir = args
    out = psr_dir / f'psr_{ipix:05d}.h5'
    if not out.exists():
        psr = _drm.get_point_source_response_per_image_pixel(ipix, _ori, coordsys='galactic',
                                                             nside_image=NSIDE_IMAGE,
                                                             nside_scatt_map=NSIDE_SCATT, earth_occ=True)
        tmp = psr_dir / f'tmp_{ipix:05d}.h5'
        psr.write(tmp, overwrite=True)
        tmp.rename(out)
    return ipix

def image_response(line):
    """Merged Galactic image response of one line (computed once, then read from data_dir)."""
    psr_dir = data_dir / f'psr_{line}_DC4_earthocc_nside{NSIDE_IMAGE}'
    esr_file = data_dir / f'esr_{line}_DC4_earthocc_nside{NSIDE_IMAGE}.h5'
    if not esr_file.exists():
        t0 = time.perf_counter()
        psr_dir.mkdir(exist_ok=True)
        npix = hp.nside2npix(NSIDE_IMAGE)
        todo = [(p, psr_dir) for p in range(npix) if not (psr_dir / f'psr_{p:05d}.h5').exists()]
        logger.info(f'{line}: computing {len(todo)} of {npix} point-source responses with {N_WORKERS} processes')
        if N_WORKERS > 1:
            # 'fork' lets the workers use the functions defined in this notebook (Linux; use N_WORKERS = 1 on macOS)
            with ProcessPoolExecutor(N_WORKERS, mp_context=multiprocessing.get_context('fork'),
                                     initializer=_init_worker, initargs=(response_files[line],)) as pool:
                for k, _ in enumerate(pool.map(_psr_to_file, todo, chunksize=4)):
                    if (k + 1) % max(1, len(todo) // 10) == 0:
                        logger.info(f'{k + 1} / {len(todo)} done, {time.perf_counter() - t0:.0f} s')
        else:
            _init_worker(response_files[line])
            for args in todo:
                _psr_to_file(args)
        esr = FullDetectorResponse.merge_psr_to_extended_source_response(str(psr_dir / 'psr_'), nside_image=NSIDE_IMAGE)
        esr.write(esr_file, overwrite=True)
        runtime[f'image response {line}'] = time.perf_counter() - t0
        # the per-pixel files in psr_dir are no longer needed and can be deleted
    return ExtendedSourceResponse.open(esr_file)

responses = {line: image_response(line) for line in SELECTIONS}
for line, rsp in responses.items():
    print(line, rsp.axes.labels, rsp.axes['NuLambda'].nside, rsp.axes['Ei'].edges, rsp.unit)

## 3. Data and background models

**Line-window data.** The line-window slice of each histogram has the axes of the response (Em = line window, Phi, PsiChi).

**26Al background.** The three sideband windows of the mock data are:
* low continuum, 1700-1755 keV;
* instrumental lines, 1760-1784 keV (the 1764 keV and 1779 keV lines plus continuum);
* high continuum, 1822-1885 keV.

Each is smoothed over the scattered direction (Gaussian, FWHM 30 deg, separately in every Phi bin, keeping the counts of the bin). The smoothing reduces the statistical noise of the templates: they contain more events than the line window, but not enough to be used bin by bin. The three templates are scaled by a background-only maximum-likelihood fit to the line window. `ImageDeconvolution` then refits their normalisations, within 0.5-2, together with the image. Templates from the data themselves are needed for 26Al: the line sits on a background made of the continuum and of instrumental lines whose spatial distributions in the CDS differ.

**511 keV background.** The released DC4 model of the 509-513 keV window is used at its own normalisation (`background_normalization_optimization: activate: False`). A free normalisation is degenerate with very extended emission, such as the 511 keV disk. The released model was made from the DC4 background simulation, i.e. from the same simulated background events as those in the mock data. It is therefore slightly better than a model that could be built for real data.

In [ ]:
LINE_WINDOWS = {'26Al': (1805.0, 1812.0), '511keV': (509.0, 513.0)}
SIDEBANDS_26AL = {'low': (1700.0, 1755.0), 'inst': (1760.0, 1784.0), 'high': (1822.0, 1885.0)}
BKG_SMOOTHING_FWHM = 30.0   # deg

def em_index(hist, lo):
    return int(np.flatnonzero(np.isclose(hist.axes['Em'].edges.to_value(u.keV)[:-1], lo))[0])

def em_slice(hist, lo):
    i = em_index(hist, lo)
    return hist.slice[{'Em': slice(i, i + 1)}]

data = {line: em_slice(counts[line], LINE_WINDOWS[line][0]) for line in SELECTIONS}
cds_axes = data['26Al'].axes

def smooth_psichi(values, fwhm_deg):
    """Smooth a (Phi, PsiChi) array over PsiChi (ring scheme) in every Phi bin, keeping the counts per Phi bin."""
    out = np.array([np.clip(hp.smoothing(v, fwhm=np.deg2rad(fwhm_deg), iter=1), 0, None) for v in values])
    s = out.sum(axis=1, keepdims=True)
    return out * np.where(s > 0, values.sum(axis=1, keepdims=True) / np.where(s > 0, s, 1), 0)

d26 = np.asarray(data['26Al'].contents, dtype=np.float64).reshape(-1)
raw = {k: np.asarray(em_slice(counts['26Al'], lo).contents, dtype=np.float64)[0] for k, (lo, hi) in SIDEBANDS_26AL.items()}
templates = {k: smooth_psichi(v, BKG_SMOOTHING_FWHM).reshape(-1) for k, v in raw.items()}

# background-only Poisson fit of the three amplitudes (EM iterations) to the line window
T = np.array(list(templates.values()))
a = np.full(len(T), d26.sum() / T.sum() )
for _ in range(500):
    mu = np.maximum(a @ T, 1e-30)
    a *= (T @ (d26 / mu)) / T.sum(axis=1)
bkg_models = {'26Al': {k: Histogram(cds_axes, contents=(a[i] * T[i]).reshape(cds_axes.shape))
                       for i, k in enumerate(templates)}}
for i, k in enumerate(templates):
    print(f'26Al background {k:5s}: {raw[k].sum():8.0f} sideband events, scaled to {a[i] * T[i].sum():7.0f} counts in the line window')
print(f'26Al line window: {d26.sum():.0f} events, background-only fit {(a @ T).sum():.0f}')

released = Histogram.open(bkg511_file).project('Em', 'Phi', 'PsiChi').to_dense()
bkg_models['511keV'] = {'released': Histogram(data['511keV'].axes, contents=np.asarray(released.contents, dtype=np.float64))}
print(f"511 keV line window: {data['511keV'].contents.sum():.0f} events, released background model "
      f"{bkg_models['511keV']['released'].contents.sum():.0f} counts")

In [ ]:
data_interfaces = {line: DataIF_COSI_DC2.load(name=line, event_binned_data=data[line],
                                               dict_bkg_binned_data=bkg_models[line], rsp=responses[line],
                                               coordsys_conv_matrix=None)
                   for line in SELECTIONS}

## 4. Image deconvolution: Richardson-Lucy and MREM

The four parameter files in this directory define the runs:

| file | algorithm | regularisation |
|---|---|---|
| `imagedeconvolution_parfile_DC4_26Al_RL.yml` | `RLadvanced`: acceleration (`MaxStep`, at most 10), response weighting 0.5, 2 deg smoothing of the update | 200 iterations |
| `imagedeconvolution_parfile_DC4_26Al_MREM.yml` | `MREM`: wavelet scales 4-64 deg, Fisher normalisation, progressive admission of scales, Newton line search | significance s = 4 |
| `imagedeconvolution_parfile_DC4_511keV_RL.yml` | as for 26Al, background fixed | 200 iterations |
| `imagedeconvolution_parfile_DC4_511keV_MREM.yml` | as for 26Al, background fixed, robust line search | significance s = 4 |

**Choosing the regularisation.** For RL, the number of iterations decides how much noise is deconvolved. More iterations sharpen the image and also build up noise peaks, especially at high latitudes. A data-driven choice is two-fold cross-validation: split the counts by binomial thinning into two halves with the same expectation, run on one half, score the Poisson likelihood of the other half, and vice versa. For the 26Al line this selects about 200 iterations.

For MREM, the significance `s` is an a priori choice of the false-positive rate. Each wavelet plane has about 3000 (nside 16) to 12000 (nside 32) coefficients. With s = 3, about 0.3 % of them, i.e. ten or more, are noise that passes the threshold; with s = 4 it is fewer than one per plane. A smaller `s` recovers more faint emission and also more spurious structure.

The MREM finest scale (4 deg) is about the angular resolution at these energies. Finer planes of the correction map have so little noise that a threshold relative to it admits sub-resolution structure.

The 511 keV runs use MREM's opt-in robust line search. At 511 keV the thresholded direction is often not an ascent direction when a new scale is admitted, and the default Newton search would then end the run early.

In [ ]:
def deconvolve(line, parfile):
    """Run one parameter file on one line; returns the ImageDeconvolution instance (results in .results)."""
    idc = ImageDeconvolution()
    idc.set_dataset([data_interfaces[line]])
    idc.read_parameterfile(parfile)
    idc.initialize()
    t0 = time.perf_counter()
    idc.run_deconvolution()
    runtime[f'{Path(parfile).stem.replace("imagedeconvolution_parfile_DC4_", "")}'] = time.perf_counter() - t0
    return idc

runs = {}

In [ ]:
%%time
runs['26Al', 'RL'] = deconvolve('26Al', 'imagedeconvolution_parfile_DC4_26Al_RL.yml')

In [ ]:
%%time
runs['26Al', 'MREM'] = deconvolve('26Al', 'imagedeconvolution_parfile_DC4_26Al_MREM.yml')

In [ ]:
%%time
runs['511keV', 'RL'] = deconvolve('511keV', 'imagedeconvolution_parfile_DC4_511keV_RL.yml')

In [ ]:
%%time
runs['511keV', 'MREM'] = deconvolve('511keV', 'imagedeconvolution_parfile_DC4_511keV_MREM.yml')

## 5. Results

### Convergence

Log-likelihood per iteration, relative to its final value, and the background normalisations. MREM runs in stages: it starts with only the coarsest scale and admits a finer scale whenever the current stage has converged (vertical lines).

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(11, 6.5))
for j, line in enumerate(SELECTIONS):
    for alg, color in (('RL', 'C0'), ('MREM', 'C1')):
        res = runs[line, alg].results
        it = [r['iteration'] for r in res]
        ll = np.array([np.sum(r['log-likelihood']) for r in res])
        axs[0, j].plot(it, ll - ll[-1], color=color, label=alg)
        if alg == 'MREM':
            for r0, r1 in zip(res[:-1], res[1:]):
                if r1['stage'] != r0['stage']:
                    axs[0, j].axvline(r1['iteration'], color=color, lw=0.6, ls=':')
        for k in res[-1]['background_normalization']:
            axs[1, j].plot(it, [r['background_normalization'][k] for r in res], color=color,
                           ls={'low': '-', 'inst': '--', 'high': ':', 'released': '-'}[k], label=f'{alg} {k}')
    axs[0, j].set_yscale('symlog', linthresh=1)
    axs[0, j].set_title(line)
    axs[0, j].set_ylabel('log-likelihood - final')
    axs[1, j].set_ylabel('background normalisation')
    axs[1, j].set_xlabel('iteration')
    axs[0, j].legend()
    axs[1, j].legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()
for (line, alg), idc in runs.items():
    r = idc.results[-1]
    print(f"{line:7s} {alg:5s}: {r['iteration']:4d} iterations, log-likelihood {np.sum(r['log-likelihood']):.1f}")

### Images

All-sky images (Mollweide, Galactic coordinates, centred on the Galactic centre) on a square-root intensity scale, common to the two algorithms of each line.

In [ ]:
def image_of(line, alg):
    """Final image (intensity per pixel, ph/cm2/s/sr) as a ring-ordered array."""
    return np.asarray(runs[line, alg].results[-1]['model'].contents.to_value(u.Unit('cm-2 s-1 sr-1')))[:, 0]

def moll(ax, m, title, vmax):
    proj = hp.projector.MollweideProj(xsize=800)
    img = proj.projmap(m, lambda x, y, z: hp.vec2pix(NSIDE_IMAGE, x, y, z))
    img = np.where(np.isfinite(img) & (img > -1e29), img, np.nan)
    cmap = plt.get_cmap('magma').copy()
    cmap.set_bad('white')
    im = ax.imshow(np.clip(img, 0, None), origin='lower', extent=(-2, 2, -1, 1), cmap=cmap,
                   norm=PowerNorm(0.5, vmin=0, vmax=vmax))
    ax.set_axis_off()
    ax.set_title(title)
    return im

fig, axs = plt.subplots(2, 2, figsize=(12, 6.5))
for i, line in enumerate(SELECTIONS):
    vmax = np.percentile(image_of(line, 'RL'), 99.9)
    for j, alg in enumerate(('RL', 'MREM')):
        im = moll(axs[i, j], image_of(line, alg), f'{line}: {alg}', vmax)
    cb = fig.colorbar(im, ax=list(axs[i]), fraction=0.025, pad=0.01)
    cb.set_label('ph/cm2/s/sr')
plt.show()

The inner Galaxy (|l| < 90 deg, |b| < 30 deg) at a larger scale:

In [ ]:
lon = np.linspace(90, -90, 721)
lat = np.linspace(-30, 30, 241)
LON, LAT = np.meshgrid(lon, lat)

fig, axs = plt.subplots(2, 2, figsize=(12, 5.2))
for i, line in enumerate(SELECTIONS):
    vmax = np.percentile(image_of(line, 'RL'), 99.9)
    for j, alg in enumerate(('RL', 'MREM')):
        v = hp.get_interp_val(image_of(line, alg), np.where(LON < 0, LON + 360, LON), LAT, lonlat=True)
        im = axs[i, j].imshow(np.clip(v, 0, None), origin='lower', extent=(90, -90, -30, 30), cmap='magma',
                              norm=PowerNorm(0.5, vmin=0, vmax=vmax), aspect='equal')
        axs[i, j].set_title(f'{line}: {alg}')
        axs[i, j].set_xlabel('l [deg]')
        axs[i, j].set_ylabel('b [deg]')
    fig.colorbar(im, ax=list(axs[i]), fraction=0.025, pad=0.01).set_label('ph/cm2/s/sr')
plt.show()

### Longitude and latitude profiles

Flux per bin (ph/cm2/s): longitude profile of the disk (|b| < 10 deg, 6 deg bins) and latitude profile of the inner Galaxy (|l| < 30 deg, 4 deg bins).

In [ ]:
l_pix, b_pix = hp.pix2ang(NSIDE_IMAGE, np.arange(hp.nside2npix(NSIDE_IMAGE)), lonlat=True)
l_pix = np.where(l_pix > 180, l_pix - 360, l_pix)
pixarea = hp.nside2pixarea(NSIDE_IMAGE)

def profile(m, kind):
    flux = m * pixarea                                   # ph/cm2/s per pixel
    if kind == 'lon':
        edges, sel, x = np.arange(-180, 181, 6), np.abs(b_pix) < 10, l_pix
    else:
        edges, sel, x = np.arange(-30, 31, 4), np.abs(l_pix) < 30, b_pix
    sel = sel & (x >= edges[0]) & (x < edges[-1])
    idx = np.digitize(x[sel], edges) - 1
    return edges, np.bincount(idx, weights=flux[sel], minlength=len(edges) - 1)[:len(edges) - 1]

fig, axs = plt.subplots(2, 2, figsize=(12, 6.5))
for i, line in enumerate(SELECTIONS):
    for j, kind in enumerate(('lon', 'lat')):
        for alg, color in (('RL', 'C0'), ('MREM', 'C1')):
            e, p = profile(image_of(line, alg), kind)
            axs[i, j].step(0.5 * (e[1:] + e[:-1]), p, where='mid', color=color, label=alg)
        axs[i, j].set_title(f'{line}: ' + ('longitude profile, |b| < 10 deg' if kind == 'lon' else 'latitude profile, |l| < 30 deg'))
        axs[i, j].set_xlabel('l [deg]' if kind == 'lon' else 'b [deg]')
        axs[i, j].set_ylabel('flux per bin [ph/cm2/s]')
        if kind == 'lon':
            axs[i, j].set_xlim(180, -180)
        axs[i, j].legend()
plt.tight_layout()
plt.show()

### Fluxes of a few regions

The flux of a region is the sum of the image over its pixels. The table compares the two algorithms. These fluxes have no error bars; a bootstrap (re-running on Poisson realisations of the data) would give them. For quantitative fluxes of extended emission, a likelihood fit of templates is more reliable than integrating an image.

In [ ]:
REGIONS = {'inner Galaxy (|l|<30, |b|<10)': (np.abs(l_pix) < 30) & (np.abs(b_pix) < 10),
           'disk (|b|<10)': np.abs(b_pix) < 10,
           'high latitude (|b|>30)': np.abs(b_pix) > 30,
           'all sky': np.ones_like(b_pix, dtype=bool)}

for line in SELECTIONS:
    print(f'{line}: flux [ph/cm2/s]')
    for name, sel in REGIONS.items():
        f = {alg: np.sum(image_of(line, alg)[sel]) * pixarea for alg in ('RL', 'MREM')}
        print(f"  {name:32s} RL {f['RL']:.2e}   MREM {f['MREM']:.2e}")

MREM shows only structure that is significant at some scale. Its images have much less high-latitude flux than RL, at the price of faint diffuse emission that is not significant at any single scale. RL recovers more of the faint emission, but turns part of the noise into a patchy haze. Which image is preferable depends on the question: MREM for a conservative map of what is significant, RL (with a data-driven stopping point) for the faint diffuse emission.

### Possible improvements

* A finer image grid. The nside 32 responses take four times longer to compute. The DC4 line responses are binned at nside 16 in the scattered direction, but the image pixels can be finer.
* Errors from a bootstrap, and null tests: run the same pipeline on the DC4 background simulation alone to see how much spurious flux the background model and the algorithm create.
* Other algorithms in `cosipy.image_deconvolution`, for example `MAP_RL` with a smoothness prior.

In [ ]:
print('measured run times [min]:')
for k, v in runtime.items():
    print(f'  {k:28s} {v / 60:6.1f}')